# TrustRAG on ColabFallback runtime. The main development machine is a local RTX 3060, and thisnotebook exists for anything that needs more VRAM than 6 GB.**Before running anything:** `Runtime` -> `Change runtime type` -> `T4 GPU`.Run the cells top to bottom. The whole setup takes about three minutes.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csvimport torchprint("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())if not torch.cuda.is_available():    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Install what Colab is missingColab already ships torch with CUDA, so we leave it alone and only add the rest.

In [ ]:
%pip install -q transformers spacy scikit-learn pyarrow!python -m spacy download en_core_web_sm -qprint("done")

## 3. Get the datasetsSame four files as the local `data/` folder, pulled straight from Hugging Face.About 58 MB. Skips anything already downloaded, so it is safe to rerun.

In [ ]:
import json, os, sys, urllib.requestROOT = "/content/data"WANT = [    ("wandb/RAGTruth-processed", "default",    "train",      "ragtruth/train.parquet"),    ("wandb/RAGTruth-processed", "default",    "test",       "ragtruth/test.parquet"),    ("pminervini/HaluEval",      "qa",         "data",       "halueval/qa.parquet"),    ("hotpotqa/hotpot_qa",       "distractor", "validation", "hotpotqa/distractor_validation.parquet"),]API = "https://datasets-server.huggingface.co/parquet?dataset="def listing(ds):    with urllib.request.urlopen(API + ds.replace("/", "%2F"), timeout=60) as r:        return json.load(r)["parquet_files"]def fetch(url, dest, expected):    if os.path.exists(dest) and abs(os.path.getsize(dest) - expected) < 1024:        print("    already have", os.path.basename(dest)); return    os.makedirs(os.path.dirname(dest), exist_ok=True)    done = 0    with urllib.request.urlopen(url, timeout=120) as r, open(dest + ".part", "wb") as f:        while True:            chunk = r.read(1 << 20)            if not chunk: break            f.write(chunk); done += len(chunk)            sys.stdout.write("\r    %s %.1f MB" % (os.path.basename(dest), done / 1e6))    os.replace(dest + ".part", dest)    print("\r    saved %s (%.1f MB)" % (os.path.basename(dest), done / 1e6))cache = {}for ds, cfg, split, rel in WANT:    cache.setdefault(ds, listing(ds))    hits = [f for f in cache[ds] if f["config"] == cfg and f["split"] == split]    print("%s [%s/%s]" % (ds, cfg, split))    for i, f in enumerate(hits):        p = os.path.join(ROOT, rel if len(hits) == 1                         else rel.replace(".parquet", "_%04d.parquet" % i))        fetch(f["url"], p, f.get("size", 0))

## 4. Verify the row counts match the proposal

In [ ]:
import pyarrow.parquet as pq, osEXPECT = {    "ragtruth/train.parquet": 15090,    "ragtruth/test.parquet": 2700,    "halueval/qa.parquet": 10000,    "hotpotqa/distractor_validation.parquet": 7405,}for rel, want in EXPECT.items():    n = pq.ParquetFile(os.path.join(ROOT, rel)).metadata.num_rows    print("%-42s %8s rows  %s" % (rel, format(n, ","), "ok" if n == want else "MISMATCH"))

## 5. Mount Drive for checkpointsColab sessions die without warning. Long scoring runs write progress here so theycan resume instead of restarting.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')CKPT = "/content/drive/MyDrive/trustrag_checkpoints"import os; os.makedirs(CKPT, exist_ok=True)print("checkpoints ->", CKPT)

## 6. Smoke-test the NLI modelDownloads about 1.7 GB the first time. The three test pairs should come backentailment, contradiction, neutral. If they do, the verifier works.

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModelForSequenceClassificationNLI = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"tok = AutoTokenizer.from_pretrained(NLI)model = AutoModelForSequenceClassification.from_pretrained(NLI).cuda().eval()print("labels:", model.config.id2label)premise = "Arthur's Magazine was started in 1844."tests = [    ("Arthur's Magazine was founded in 1844.", "entailment"),    ("Arthur's Magazine was founded in 1901.", "contradiction"),    ("Arthur's Magazine was printed in Philadelphia.", "neutral"),]with torch.no_grad():    for hyp, want in tests:        x = tok(premise, hyp, return_tensors="pt", truncation=True).to("cuda")        probs = torch.softmax(model(**x).logits[0], -1)        got = model.config.id2label[int(probs.argmax())]        flag = "ok" if got.lower() == want else "unexpected"        print("%-48s -> %-14s %s" % (hyp[:48], got, flag))print("\npeak VRAM: %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9))

## 7. Throughput checkTells you how long a full RAGTruth pass will take, so you can size the runbefore starting it.

In [ ]:
import time, torchpairs = [("Arthur's Magazine was started in 1844.",          "Arthur's Magazine was founded in 1844.")] * 256BS = 32torch.cuda.synchronize(); t0 = time.time()with torch.no_grad():    for i in range(0, len(pairs), BS):        batch = pairs[i:i + BS]        x = tok([p for p, _ in batch], [h for _, h in batch],                return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")        model(**x)torch.cuda.synchronize()dt = time.time() - t0rate = len(pairs) / dtprint("%.1f pairs/sec at batch size %d" % (rate, BS))print("15,090 responses x ~8 claims x ~10 evidence sentences = ~1.2M pairs")print("estimated full pass: %.1f hours" % (1_200_000 / rate / 3600))